# HMIE Batch Validation

Validate HMIE/Scale datasets using `databridge`. Point this notebook at a
directory containing one or more batches and it will check:

- **Folder structure** -- snippet directories with `seq_*/` video containers
- **Video integrity** -- each video opens and decodes representative frames
- **Annotation coverage** -- every annotation pairs to a video (and vice-versa)
- **Scale spec compliance** -- annotation JSON conforms to the Scale Video Playback schema

Set `CHECK_VIDEO = False` below to skip video decoding for a faster JSON-only run.

In [ ]:
from pathlib import Path

import databridge

print(f"databridge {databridge.__version__}")

# -- Configuration --------------------------------------------------------
BATCH_ROOT = Path("/path/to/sunet/hmie/data")  # root containing batch(es)

# CHECK_VIDEO controls the per-file video probe (opening each .mp4 with
# OpenCV and decoding sample frames to detect corruption, dropped frames,
# wrong FPS, etc.). It does NOT control whether videos must exist:
# annotation/video pair discovery (orphan_annotation, orphan_video,
# multiple_videos_in_seq_mp4) and folder-structure checks always run.
#
# Set False for a fast JSON-only sweep (annotation schema + Scale spec
# compliance + folder layout). Set True for the full integrity scan.
CHECK_VIDEO = True

## Run validation

The HMIE dataset is organised as one or more *batches* under a root folder.
`validate_batches` walks that root, runs the full validation pipeline on each
batch, and yields `(batch_path, ValidationResult)` pairs as it goes -- so the
caller can stream results without buffering tens of thousands of batches.

Each `ValidationResult` covers the four requirement checks:

- folder structure (snippet dirs, `seq_*/` containers)
- annotation coverage (every annotation paired to a video, no orphans)
- Scale spec compliance (annotation JSON against the Scale Video Playback schema)
- video integrity (per-file decode probe, only when `check_video_integrity=True`)

`result.summary()` prints a short pass/fail block per batch; the result object
itself carries the full finding list, per-check counts, and label histogram
for downstream programmatic use.

In [ ]:
from databridge import ValidationResult, validate_batches

results: list[tuple[Path, ValidationResult]] = []

for batch, result in validate_batches(BATCH_ROOT, check_video_integrity=CHECK_VIDEO):
    results.append((batch, result))
    print(f"--- {batch.name} ---")
    print(result.summary())

print(f"Validated {len(results)} batch(es)")

## Roll the results up

A short cross-batch summary: how many batches passed, total errors and
warnings, and a per-batch pass/fail line with snippet and finding counts.

In [ ]:
if not results:
    print("No batches produced results.")
else:
    passed = sum(1 for _, r in results if r.passed)
    total_errs = sum(sum(r.finding_severity_counts["error"].values()) for _, r in results)
    total_warns = sum(sum(r.finding_severity_counts["warning"].values()) for _, r in results)
    print(f"{passed}/{len(results)} batches passed")
    print(f"{total_errs:,} errors, {total_warns:,} warnings total\n")

    for batch, r in results:
        tag = "PASS" if r.passed else "FAIL"
        errs = sum(r.finding_severity_counts["error"].values())
        warns = sum(r.finding_severity_counts["warning"].values())
        print(f"  {tag}  {batch.name}  ({r.snippet_count:,} snippets, {errs:,} errors, {warns:,} warnings)")

## Generate an HTML report

For a shareable artifact (review, hand-off, attaching to a ticket),
`render_html_report` turns the per-batch results into a single self-contained
HTML file: the four requirement-check indicators, per-check finding counts,
and the label histogram for each batch. Open the output path in a browser --
no server, no extra assets.

In [ ]:
from databridge import render_html_report

# One HTML file per batch, written next to the notebook. Rename / route
# the output directory however you like -- render_html_report returns a
# self-contained string, so the file write is entirely up to the caller.
report_dir = Path("databridge-reports")
report_dir.mkdir(exist_ok=True)

for batch, result in results:
    out = report_dir / f"{batch.name}.html"
    out.write_text(render_html_report(result), encoding="utf-8")
    print(f"  {out}")